### Homework 5: Question search engine

Remeber week01 where you used GloVe embeddings to find related questions? That was.. cute, but far from state of the art. It's time to really solve this task using context-aware embeddings.

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [0]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/a9/b6/5257d04ae327b44db31f15cce39e6020cc986333c715660b1315a9724d82/transformers-4.51.3-py3-none-any.whl.metadata
  Obtaining dependency information for datasets from https://files.pythonhosted.org/packages/20/34/a08b0ee99715eaba118cbe19a71f7b5e2425c2718ef96007c325944a1152/datasets-3.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for accelerate from https://files.pythonhosted.org/packages/f8/bb/be8146c196ad6e4dec78385d91e92591f8a433576c4e04c342a636fcd811/accelerate-1.7.0-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/1.5 MB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 56.5 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Obtaining dependency information for huggingface-hub<1.0,>=0.30.0 from https://files.pythonhosted.org/packages/83/81/a8fd9c226f7e3

Exception ignored on calling ctypes callback function: <function _ThreadpoolInfo._find_modules_with_dl_iterate_phdr.<locals>.match_module_callback at 0x7fbea3affa60>
Traceback (most recent call last):
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 400, in match_module_callback
    self._make_module_from_path(filepath)
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 515, in _make_module_from_path
    module = module_class(filepath, prefix, user_api, internal_api)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 606, in __init__
    self.version = self.get_version()
                   ^^^^^^^^^^^^^^^^^^
  File "/databricks/python/lib/python3.11/site-packages/threadpoolctl.py", line 646, in get_version
    config = get_config().split()
             ^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'split'
20

[2025-05-15 12:44:04,192] [INFO] [real_accelerator.py:239:get_accelerator] Setting ds_accelerator to cuda (auto detect)


/usr/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/usr/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


### Load data and model

In [0]:
qqp = datasets.load_dataset('SetFit/qqp')
print('\n')
print("Sample[0]:", qqp['train'][0])
print("Sample[3]:", qqp['train'][3])

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:45: UserWarning: The cache_dir for this dataset is /root/.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [DBFS].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/dbfs/']
  warnings.warn(warning_message)


README.md:   0%|          | 0.00/313 [00:00<?, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.
/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:14: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(
Repo card metadata block was not found. Setting CardData to empty.


train.jsonl:   0%|          | 0.00/70.8M [00:00<?, ?B/s]

validation.jsonl:   0%|          | 0.00/7.83M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/76.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/363846 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/40430 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/390965 [00:00<?, ? examples/s]



Sample[0]: {'text1': 'How is the life of a math student? Could you describe your own experiences?', 'text2': 'Which level of prepration is enough for the exam jlpt5?', 'label': 0, 'idx': 0, 'label_text': 'not duplicate'}
Sample[3]: {'text1': 'What can one do after MBBS?', 'text2': 'What do i do after my MBBS ?', 'label': 1, 'idx': 3, 'label_text': 'duplicate'}


In [0]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/320 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

### Tokenize the data

In [0]:
MAX_LENGTH = 128
def preprocess_function(examples, tokenizer):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed = qqp.map(lambda x: preprocess_function(x, tokenizer=tokenizer), batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/433M [00:00<?, ?B/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [0]:
qqp_preprocessed['train'][0]#['input_ids']

{'text1': 'How is the life of a math student? Could you describe your own experiences?',
 'text2': 'Which level of prepration is enough for the exam jlpt5?',
 'label': 0,
 'idx': 0,
 'label_text': 'not duplicate',
 'input_ids': [101,
  1731,
  1110,
  1103,
  1297,
  1104,
  170,
  12523,
  2377,
  136,
  7426,
  1128,
  5594,
  1240,
  1319,
  5758,
  136,
  102,
  5979,
  1634,
  1104,
  3073,
  20488,
  2116,
  1110,
  1536,
  1111,
  1103,
  12211,
  179,
  1233,
  6451,
  1571,
  136,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,

In [0]:
print(repr(qqp_preprocessed['train'][0]['input_ids'])[:100], "...")

[101, 1731, 1110, 1103, 1297, 1104, 170, 12523, 2377, 136, 7426, 1128, 5594, 1240, 1319, 5758, 136,  ...


### Task 1: evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [0]:
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=2, shuffle=False, collate_fn=transformers.default_data_collator
)

In [0]:
for batch in val_loader:
     break  # here be your training code
print("Sample batch:", batch)

with torch.no_grad():
  predicted = model(
      input_ids=batch['input_ids'],
      attention_mask=batch['attention_mask'],
      token_type_ids=batch['token_type_ids']
  )

print('\nPrediction (probs):', torch.softmax(predicted.logits, dim=1).data.numpy())

Sample batch: {'labels': tensor([0, 0]), 'idx': tensor([0, 1]), 'input_ids': tensor([[  101,  2009,  1132,  2170,   118,  4038,  1177,  2712,   136,   102,
          2009,  1132,  1117, 10224,  4724,  1177,  2712,   136,   102,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,    

In [0]:
(torch.argmax(torch.softmax(predicted.logits, dim=1), dim=1) == batch['labels']).sum() / len(batch['labels'])

tensor(1.)

__Your task__ is to measure the validation accuracy of your model.
Doing so naively may take several hours. Please make sure you use the following optimizations:

- run the model on GPU with no_grad
- using batch size larger than 1
- use optimize data loader with num_workers > 1
- (optional) use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [0]:
from tqdm import tqdm
# <A whole lot of YOUR CODE HERE>
# ...


# accuracy = <Validation accuracy, between 0 and 1>

DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 25
NUM_WORKERS = 1 # does not have impact here...

print(f"{DEVICE=}\n{BATCH_SIZE=}")

val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=NUM_WORKERS
)

DEVICE='cuda:0'
BATCH_SIZE=25


In [0]:
def evaluate_accuracy(model, data_loader, batch_size=BATCH_SIZE, loss_func=None, eval_share=1, verbose=False):
    model.eval()
    model.to(DEVICE)
    numerator = 0
    denominator = 0
    running_loss = 0
    n_batches = 0

    total_batches = len(data_loader)
    n_required_batches = int(total_batches * eval_share)
    
    if verbose:
        print(f"\tEvaluating on {n_required_batches} out of {total_batches} batches...")
    
    for i, batch in enumerate(tqdm(data_loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with torch.no_grad():
            predicted = model(
                input_ids=batch['input_ids'],
                attention_mask=batch['attention_mask'],
                token_type_ids=batch['token_type_ids']
            )
            sm = torch.softmax(predicted.logits, dim=1)
            pred_labels = torch.argmax(sm, dim=1)

            numerator += (pred_labels == batch['labels']).sum()
            denominator += len(batch['labels'])

            if loss_func is not None:
                loss = loss_func(predicted.logits, batch['labels'])
                running_loss += loss.item()
                n_batches += 1
        
        # only evaluating based on the required share of data
        if i >= n_required_batches:
            break

    accuracy = (numerator / denominator).cpu().item()
    loss = running_loss / n_batches

    return accuracy, loss

loss_func = nn.CrossEntropyLoss()
accuracy, loss = evaluate_accuracy(model, val_loader, loss_func=loss_func, eval_share=0.1, verbose=True)
print(f"Accuracy: {accuracy}. Loss: {loss}")

	Evaluating on 161 out of 1618 batches...


  0%|          | 0/1618 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
 10%|▉         | 161/1618 [00:23<03:34,  6.78it/s]


Accuracy: 0.9101234674453735. Loss: 0.3723042540965073


In [0]:
loss

0.3723042540965073

In [0]:
assert 0.9 < accuracy < 0.91

### Task 2: train the model (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

In [0]:
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 50
NUM_WORKERS = 1
LOSS_ACCUMULATION_ITERATIONS = 10

# on this share the model will be evaluated from time to time during training
VALIDATION_EVAL_SHARE = 0.1

print(f"{DEVICE=}\n{BATCH_SIZE=}")

DEVICE='cuda:0'
BATCH_SIZE=50


In [0]:
# model_name = "bert-base-uncased"
model_name = "google-bert/bert-base-cased"
# model_name = "FacebookAI/roberta-base"
tokenizer_my = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at google-bert/bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [0]:
MAX_LENGTH = 128
def preprocess_function(examples, tokenizer):
    result = tokenizer(
        examples['text1'], examples['text2'],
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result['label'] = examples['label']
    return result

qqp_preprocessed_my = qqp.map(lambda x: preprocess_function(x, tokenizer=tokenizer_my), batched=True)

Map:   0%|          | 0/363846 [00:00<?, ? examples/s]

Map:   0%|          | 0/40430 [00:00<?, ? examples/s]

Map:   0%|          | 0/390965 [00:00<?, ? examples/s]

In [0]:
# freezing BERT layers
for param in model.bert.parameters():
    # param.requires_grad = False
    param.requires_grad = True

for param in model.classifier.parameters():
    param.requires_grad = True

_trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
_total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters = {_trainable_params}, which is {_trainable_params / _total_params * 100}% of total parameters")

Trainable parameters = 108311810, which is 100.0% of total parameters


In [0]:
train_set = qqp_preprocessed_my['train']
train_loader = torch.utils.data.DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=transformers.default_data_collator,
    num_workers=NUM_WORKERS
)

val_set = qqp_preprocessed_my['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False, collate_fn=transformers.default_data_collator,
    num_workers=NUM_WORKERS
)

print(f"{len(train_loader)=} batches x {BATCH_SIZE} obs each\n{len(val_loader)=} batches x {BATCH_SIZE} obs each")

len(train_loader)=7277 batches x 50 obs each
len(val_loader)=809 batches x 50 obs each


In [0]:
train_set[0]

{'text1': 'How is the life of a math student? Could you describe your own experiences?',
 'text2': 'Which level of prepration is enough for the exam jlpt5?',
 'label': 0,
 'idx': 0,
 'label_text': 'not duplicate',
 'input_ids': [101,
  1731,
  1110,
  1103,
  1297,
  1104,
  170,
  12523,
  2377,
  136,
  7426,
  1128,
  5594,
  1240,
  1319,
  5758,
  136,
  102,
  5979,
  1634,
  1104,
  3073,
  20488,
  2116,
  1110,
  1536,
  1111,
  1103,
  12211,
  179,
  1233,
  6451,
  1571,
  136,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'token_type_ids': [0,
  0,
  0,

In [0]:
from tqdm import tqdm
opt = torch.optim.AdamW(model.parameters(), lr=2e-5)
loss_func = nn.CrossEntropyLoss()

model.train(True)
model.to(DEVICE)

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [0]:
import random
import numpy as np

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

In [0]:
stats = {}

for epoch in range(10):
    print(f"Epoch {epoch}...")

    # setting training behavior for Droppout & other layers
    model.train(True)

    epoch_stats = {
        "numerator": 0,
        "denominator": 0,
    }
    running_training_loss = 0
    n_training_batches = 0
    for i, batch in enumerate(tqdm(train_loader)):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        predicted = model(
            input_ids=batch['input_ids'],
            attention_mask=batch['attention_mask'],
            token_type_ids=batch['token_type_ids']
        )
        # accumulating loss and stepping once in LOSS_ACCUMULATION_ITERATIONS
        loss = loss_func(predicted.logits, batch['labels'])

        # opt stepping
        loss_per_it = loss / LOSS_ACCUMULATION_ITERATIONS
        loss_per_it.backward()
        if (i + 1) % LOSS_ACCUMULATION_ITERATIONS == 0 or (i + 1) == len(train_loader):
            opt.step()
            opt.zero_grad()

        # measuring the accuracy
        sm = torch.softmax(predicted.logits, dim=1)
        pred_labels = torch.argmax(sm, dim=1)
        numerator = (pred_labels == batch['labels']).sum()
        denominator = len(batch['labels'])
        epoch_stats['numerator'] += numerator
        epoch_stats['denominator'] += denominator

        # measuring the loss
        running_training_loss += loss.item()
        n_training_batches += 1

        if i % 100 == 0:
            _intermediate_train_accuracy = (epoch_stats['numerator'] / epoch_stats['denominator']).cpu().item()
            _intermediate_train_loss = (running_training_loss / n_training_batches)
            print(f"{_intermediate_train_loss=}, {_intermediate_train_accuracy=}")
    
    train_accuracy = (epoch_stats['numerator'] / epoch_stats['denominator']).cpu().item()
    eval_accuracy, eval_loss = evaluate_accuracy(
        model, val_loader, loss_func=loss_func, eval_share=VALIDATION_EVAL_SHARE, verbose=True
    )
    epoch_stats['train_accuracy'] = train_accuracy
    epoch_stats['eval_accuracy'] = eval_accuracy
    stats[epoch] = epoch_stats

    avg_training_loss = running_training_loss / n_training_batches

    print(f"Train loss: {avg_training_loss.item():.4f}, Eval loss: {eval_loss.item():.4f}\n)")
    print(f"Train accuracy: {train_accuracy:.4f}, Eval accuracy: {eval_accuracy:.4f}")

Epoch 0...


  0%|          | 0/7277 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
  0%|          | 1/7277 [00:00<1:59:09,  1.02it/s]

_intermediate_train_loss=0.6865487098693848, _intermediate_train_accuracy=0.5199999809265137


  1%|▏         | 101/7277 [01:22<1:37:59,  1.22it/s]

_intermediate_train_loss=0.6095852916783625, _intermediate_train_accuracy=0.6493068933486938


...

_intermediate_train_loss=0.36664229284686106, _intermediate_train_accuracy=0.8267332911491394


 56%|█████▋    | 4101/7277 [55:55<43:25,  1.22it/s]

_intermediate_train_loss=0.3649301070304825, _intermediate_train_accuracy=0.827861487865448


 57%|█████▋    | 4116/7277 [56:07<43:00,  1.23it/s]

com.databricks.backend.common.rpc.CommandCancelledException
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$5(SequenceExecutionState.scala:136)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3(SequenceExecutionState.scala:136)
	at com.databricks.spark.chauffeur.SequenceExecutionState.$anonfun$cancel$3$adapted(SequenceExecutionState.scala:133)
	at scala.collection.immutable.Range.foreach(Range.scala:158)
	at com.databricks.spark.chauffeur.SequenceExecutionState.cancel(SequenceExecutionState.scala:133)
	at com.databricks.spark.chauffeur.ExecContextState.cancelRunningSequence(ExecContextState.scala:728)
	at com.databricks.spark.chauffeur.ExecContextState.$anonfun$cancel$1(ExecContextState.scala:446)
	at scala.Option.getOrElse(Option.scala:189)
	at com.databricks.spark.chauffeur.ExecContextState.cancel(ExecContextState.scala:446)
	at com.databricks.spark.chauffeur.ExecutionContextManagerV1.can

In [0]:
accuracy, loss = evaluate_accuracy(
    model, val_loader, loss_func=loss_func, eval_share=VALIDATION_EVAL_SHARE, verbose=True
)
print(f"Accuracy: {accuracy}")

	Evaluating on 80 out of 809 batches...


  0%|          | 0/809 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
 10%|▉         | 80/809 [00:23<03:38,  3.34it/s]

Accuracy: 0.872592568397522


### Task 3: try the full pipeline (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

In [0]:
# will use the default model for this part as I didn't save the state_dict of the fine tune :(
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [0]:
def find_unique_questions(dataset):
    questions = []
    for obs in dataset:
        questions.append(obs['text1'])
        questions.append(obs['text2'])
    
    return list(set(questions))

In [0]:
UNIQUE_TRAIN_QUESTIONS = find_unique_questions(train_set)
print(len(UNIQUE_TRAIN_QUESTIONS))
UNIQUE_TRAIN_QUESTIONS[:2]

493874


['What are best IAS coaching institutes in Mumbai?',
 'If you had children, would you rather they use Trump or Hillary as role model to learn behavior or as a moral mentor?']

In [0]:
MAX_LENGTH = 128
def tokenize_paired_questions(question, question_another, tokenizer):
    result = tokenizer(
        question, question_another,
        padding='max_length', max_length=MAX_LENGTH, truncation=True
    )
    result = {k: torch.as_tensor(v) for k, v in result.items()}
    return result

tokenize_paired_questions('why?', 'can I?', tokenizer)

{'input_ids': tensor([ 101, 1725,  136,  102, 1169,  146,  136,  102,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0]),
 'token_type_ids': tensor([0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0

In [0]:
from torch.utils.data import Dataset, DataLoader

class MyDataset(Dataset):
    def __init__(self, question, tokenizer, all_questions=UNIQUE_TRAIN_QUESTIONS):
        self.question = question
        self.all_questions = all_questions
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.all_questions)

    def __getitem__(self, idx):
        target_question = self.all_questions[idx]
        encoded = tokenize_paired_questions(self.question, target_question, self.tokenizer)
        return encoded

In [0]:
paired_questions_dataset = MyDataset(
    question='How is the life of a math student?',
    tokenizer=tokenizer,
    all_questions=UNIQUE_TRAIN_QUESTIONS,
)

In [0]:
BATCH_SIZE = 50
SHUFFLE = True
NUM_WORKERS = 10

paired_questions_dataloader = DataLoader(
    paired_questions_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE, num_workers=NUM_WORKERS
)

In [0]:
def get_candidates(model, paired_questions_dataloader, all_questions=UNIQUE_TRAIN_QUESTIONS):
    DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model.to(DEVICE)
    model.eval()

    candidates = []

    with torch.no_grad():
        for i, batch in enumerate(tqdm(paired_questions_dataloader)):

            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            outputs = model(**batch)

            probs = torch.softmax(outputs.logits, dim=-1)  
            preds = probs.argmax(dim=-1)
            
            # Find duplicates (class=1)
            is_duplicate = (preds == 1)
            if is_duplicate.any():
                # Decode input_ids for duplicate candidates
                duplicate_indices = torch.nonzero(is_duplicate).flatten()
                for idx in duplicate_indices:
                    # Decode the paired question (text2) from input_ids
                    input_ids = batch["input_ids"][idx].cpu()
                    decoded_question = tokenizer.decode(input_ids, skip_special_tokens=True)
                    
                    # Extract the second question (assumes format: [CLS] Q1 [SEP] Q2 [SEP])
                    decoded_parts = decoded_question.split("[SEP]")
                    if len(decoded_parts) >= 2:
                        candidate_question = decoded_parts[1].strip()
                        prob = probs[idx][1].item()
                        candidates.append((candidate_question, prob))
            
            del batch, outputs, probs, preds
            torch.cuda.empty_cache()

            if i % 200 == 0:
                print(f"{i}.{candidates=}\n")

    candidates.sort(key=lambda x: -x[1])  
    top_5_duplicates = candidates[:5]

    return top_5_duplicates

__Bonus:__ for bonus points, try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

In [0]:
def questions_jaccard_similarity(question1, question2):
    """Jaccard similarity of word sets"""
    words1 = set(question1.lower().split())
    words2 = set(question2.lower().split())
    intersection = len(words1 & words2)
    union = len(words1 | words2)
    return intersection / union if union > 0 else 0

In [0]:
QUESTION = 'What is the life of a female IAS officer?'
base_scores = [(i, questions_jaccard_similarity(QUESTION, q)) for i, q in enumerate(UNIQUE_TRAIN_QUESTIONS)]
base_scores = sorted(base_scores, key=lambda x: -x[1])
idx_top100 = [el[0] for el in base_scores][:5]

In [0]:
preliminary_list = [UNIQUE_TRAIN_QUESTIONS[ix] for ix in idx_top100]
preliminary_list[:5]

['What is the life of a female IAS officer?',
 'How is the life of a female IAS officer?',
 'What is the life of a RBI B grade officer?',
 'What is the daily life of a librarian?',
 'What is the work of an IAS officer?']

In [0]:
QUESTION

'What is the life of a female IAS officer?'

In [0]:
paired_questions_dataset = MyDataset(
    question=QUESTION,
    tokenizer=tokenizer,
    all_questions=preliminary_list,
)

BATCH_SIZE = 1
SHUFFLE = True
NUM_WORKERS = 1

paired_questions_dataloader = DataLoader(
    paired_questions_dataset, batch_size=BATCH_SIZE, shuffle=SHUFFLE, num_workers=NUM_WORKERS
)

In [0]:
res = get_candidates(model, paired_questions_dataloader, all_questions=preliminary_list)
res

  0%|          | 0/5 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 5/5 [00:00<00:00, 24.39it/s]

0.candidates=[]



[]

In [0]:
tokenizer.decode(paired_questions_dataset[0]['input_ids'])

'[CLS] What is the life of a female IAS officer? [SEP] What is the life of a female IAS officer? [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]'